# 팀원 통합 RSS 수집기 검증

`team_rss_news/src/rss_pipeline.py`(19개 매체·36개 피드, RSS → 매체별 본문 추출 → SQLite)를 그대로 실행한 결과가 믿을 만한지 확인합니다.

**방법**
1. 팀원 단위 테스트 실행
2. 팀원 수집기를 **수정 없이** 1회 실행 (매체별 본문 5건, 결과는 `rss_live/cache/team_check/`에 따로 저장)
3. 같은 기사를 독립적으로 다시 받아 추출해서 대조
   - 기준 본문: 언론사가 페이지에 넣어둔 JSON-LD `articleBody`(있으면), 없으면 trafilatura 추출 본문 (`2_rss_history_test.ipynb`와 같은 함수)
   - `team_recall`: 기준 본문 중 팀원 본문에 들어있는 비율 → 낮으면 **본문 누락**
   - `team_precision`: 팀원 본문 중 기준 본문에 들어있는 비율 → 낮으면 **기사 아닌 내용 섞임**
   - 페이지 판정 대조: 차단·유료·영상·기사 아님 페이지를 팀원 수집기가 본문으로 저장했는지
   - 반복·비대 검사: 같은 문단이 10% 넘게 반복되거나 기준 본문보다 3배 넘게 길면 `DUPLICATED_OR_BLOATED` (겹침 비율은 같은 문장이 여러 번 나와도 한 번으로 세므로 따로 검사)
4. 피드 수집·저장 점검: 날짜 형식, 중복, 상태값 분포, 수집 간격, 매체 간 중복 기사
5. 다른 방식과 비교: 같은 기사에서 팀원 선택자 / trafilatura / JSON-LD 본문 / RSS 본문

팀원 코드는 수정하지 않습니다. 팀원 코드는 원래 `rss_news/` 폴더에 있었고, 이 노트북을 다시 돌리는 데 필요한 것(수집기·본문 규칙·설정·테스트·선정 문서)만 `team_rss_news/`로 옮겨 두었습니다.

## 팀원 언론사 선정 과정 (문서 기준 정리)

| 단계 | 팀원이 한 것 | 근거 문서 |
|---|---|---|
| 출발 매체 | 이전 인수인계 자료의 6개 매체 (CNBC, Fortune, Motley Fool, Zacks, Fox Business, Business Insider) | `docs/rss_media_status.md` |
| 확대 후보 | "일반·경제·산업·기술 뉴스를 넓게" 받는 영문 RSS 25개 매체 조사 (BBC, Guardian, NYT, WSJ, Bloomberg 등) | `docs/rss_expansion_candidates.md`, `docs/rss_expansion_research_20261029.md` |
| RSS 판정 | HTTP 200 + RSS/Atom 분석 성공 + 최근 14일 안 기사. 공식 RSS 안내 여부는 별도 기록 | 같은 문서 |
| 본문 판정 | 매체별 표본 1~3건 원문 추출 → 사용자가 원문과 직접 대조 → 확인된 홍보·추천 문구를 정제 규칙으로 추가 | `docs/article_body_pilot.md`, `docs/media_review_decisions.md` |
| 제외 | 원문 401/403(WSJ, Bloomberg, NYT, FT, MarketWatch 등), 구독 제한으로 대조 불가(The Verge, MIT), 갱신 멈춤(Fast Company), 사용자 결정(WIRED) | `docs/rss_media_review_summary_20261104.md` |

**검토 의견** (아래 실행 결과와 함께 판단)
- 기준이 "RSS가 되는가 / 본문이 공개되는가"뿐이고 **증권·금융 관련성과 언론 신뢰도는 기준에 없음** → Motley Fool·Zacks(투자 조언·자동 생성 기사), Al Jazeera·ProPublica·Engadget 등 금융과 거리가 먼 매체가 포함됨
- 본문 판정 표본이 매체당 1~3건이라 작음 → 이 노트북에서 매체당 최대 20건으로 다시 확인
- 후보를 넓게 찾지 않아 같은 매체의 경제 섹션(CNBC Investing·Earnings, ABC·NBC Business 등)이나 LA Times 같은 곳은 빠짐 (`3_rss_live_feed_test.ipynb` 참고)
- 차단 매체 판단(WSJ·Bloomberg·NYT 등 원문 401/403)은 `3_rss_live_feed_test.ipynb`의 독립 시험 결과와 일치

In [1]:
import calendar
import gzip
import hashlib
import html
import json
import re
import time
import unicodedata
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
from dateutil import parser as date_parser
from lxml import html as lxml_html
from trafilatura import extract as trafilatura_extract
from trafilatura.utils import decode_file
import sqlite3
import subprocess
import sys

## 0. 설정

In [2]:
BASE_DIR = Path("rss_live")
CACHE_DIR = BASE_DIR / "cache"
FEED_CACHE_DIR = CACHE_DIR / "feeds"

TEAM_RSS_DIR = Path("team_rss_news")
TEAM_CHECK_DIR = CACHE_DIR / "team_check"
TEAM_DB_PATH = TEAM_CHECK_DIR / "combined_rss.sqlite3"
TEAM_RUN_DIR = TEAM_CHECK_DIR / "runs"

# 팀원 수집기 실행 조건 (DB가 없을 때만 실행)
TEAM_BODY_PER_SOURCE = 5
TEAM_DELAY = 2

TEAM_RATE_PATH = BASE_DIR / "team_feed_rate.csv"
TEAM_METHODS_PATH = BASE_DIR / "team_methods.parquet"
TEAM_CHECK_PATH = BASE_DIR / "team_check.parquet"
TEAM_SUMMARY_PATH = BASE_DIR / "team_check_summary.csv"

# 대조 기준
MIN_TEAM_RECALL = 0.8       # 기준 본문의 80% 이상이 팀원 본문에 있어야 누락 없음
MIN_TEAM_PRECISION = 0.8    # 팀원 본문의 80% 이상이 기준 본문에 있어야 섞인 내용 없음
MAX_DUP_PARAGRAPH_RATE = 0.1  # 같은 문단이 반복되는 비율 상한 (겹침 비율은 반복을 못 잡아서 따로 검사)
MAX_LENGTH_RATIO = 3.0        # 팀원 본문이 기준 본문보다 이 배수 이상 길면 비정상

# 수집 간격 판정 (3_rss_live_feed_test.ipynb와 같음)
STALE_DAYS = 7
RATE_WINDOW_HOURS = 24
POLL_SAFETY = 0.5
MIN_POLL_MINUTES = 5
MAX_POLL_MINUTES = 360

# 팀원 문서의 권장 실행 간격(분): docs/continuous_collection.md "2시간마다", RSS_HANDOFF.md 예시 600초
TEAM_INTERVALS = {"2시간": 120, "10분": 10}

NEWS_TIMEZONE = ZoneInfo("America/New_York")
REQUEST_TIMEOUT = 30
REQUEST_DELAY = 0.5
MAX_RETRIES = 3
WAYBACK_BLOCK_WAIT = 60
MIN_VALID_DATE = datetime(1995, 1, 1, tzinfo=timezone.utc)

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36"
    ),
    "Accept": "application/rss+xml, application/atom+xml, application/xml;q=0.9, */*;q=0.8",
}

In [3]:
# ============================================================
# 본문 수집 설정 (7~9단계)
# ============================================================
# 통과 피드마다 테스트 기간 기사 중 이 수만큼 무작위 표본 (기사가 적으면 전부)
BODY_SAMPLE_PER_FEED = 40
BODY_SAMPLE_SEED = 42

# 기사 페이지는 피드보다 요청 간격을 넉넉히
BODY_FETCH_DELAY = 1.0
BODY_FETCH_RETRIES = 1

# 같은 사이트에서 연속으로 이 횟수만큼 차단(401/403/429)·연결 오류가 나면 그 사이트 나머지 기사는 요청하지 않음
BODY_HOST_FAIL_LIMIT = 3

HTML_HEADERS = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

MIN_BODY_WORDS = 80         # 본문 최소 단어 수 (짧은 단신도 통과하도록 낮게)
MIN_LD_COVERAGE = 0.6       # JSON-LD articleBody 대비 추출 본문 비율 하한
SHINGLE_SIZE = 5            # 두 본문을 비교하는 단위 (연속 5단어 묶음)
MIN_REF_COVERAGE = 0.8      # 기준 본문(JSON-LD 또는 사이트별 본문 영역)의 80% 미만만 trafilatura에 있으면 기준 본문 사용
RSS_FULL_COVERAGE = 0.9     # 원문 본문의 90% 이상이 RSS 본문에 있으면 RSS 본문 = 전문

# 사이트별 본문 영역 (태그, class 이름). trafilatura 결과를 교차 확인하는 기준 본문으로 사용
#   2026-11-05 각 사이트 기사 2건씩 HTML을 열어 <p>가 모여 있는 요소를 확인해서 정함
#   (rss_news 본문 시험 코드의 매체별 SELECTORS와 같은 방식)
#   등록된 사이트는 이 영역을 최종 본문으로 쓰고, trafilatura 결과와 겹침 비율로 서로 확인
SITE_BODY_CONTAINERS = {
    "motherjones.com": ("article", "entry-content"),
    "techcrunch.com": ("div", "entry-content"),
    "semiengineering.com": ("div", "post_cnt"),
    "nypost.com": ("div", "single__content"),
    "arstechnica.com": ("div", "post-content"),   # 광고를 사이에 두고 여러 개로 나뉨 → trafilatura가 첫 덩어리만 가져감
    "9to5mac.com": ("div", "post-content"),
    "wired.com": ("div", "body__inner-container"),
}

# 본문 영역 안에서 빼는 요소 (class 단어가 이 접두어로 시작)
JUNK_CLASS_PREFIXES = (
    "ad-", "ads", "advert", "newsletter", "related", "promo", "recirc", "share", "social", "subscribe",
    "inline-module",   # NY Post 본문 중간 뉴스레터 가입·관련 기사 박스
)

# 본문 영역 문단 중 링크 글자가 이 비율 이상이면 관련 기사·상품 링크 목록으로 보고 제외 (9to5Mac 쇼핑 목록 등)
MAX_LINK_DENSITY = 0.8

MIN_BODY_ARTICLES = 8       # 판정에 필요한 최소 기사 페이지 수
MIN_RSS_COMPARABLE = 5      # RSS 전문 여부 판정에 필요한 최소 비교 가능 기사 수
BODY_USABLE_RATE = 0.8
BODY_UNSTABLE_RATE = 0.5

HTML_DIR = CACHE_DIR / "html"
HTML_FETCH_LOG_PATH = CACHE_DIR / "html_fetch.jsonl"

BODY_CHECK_PATH = BASE_DIR / "body_check.parquet"
BODY_SUMMARY_PATH = BASE_DIR / "body_summary.csv"
BODY_REVIEW_PATH = BASE_DIR / "body_review.csv"
BODY_LISTS_PATH = BASE_DIR / "body_lists.json"

In [4]:
# ============================================================
# 페이지 판정 패턴 (1_google_rss_source_test.ipynb와 같은 기준)
# ============================================================
VIDEO_PATH_PATTERN = re.compile(
    r"/(video|videos|watch|clip|clips|v|live-tv|livetv|podcast|podcasts|audio|gallery|galleries|slideshow|photos)(/|$)",
    re.IGNORECASE,
)
VIDEO_HOST_PATTERN = re.compile(r"^(video|videos|watch|tv)\.", re.IGNORECASE)
LIVE_BLOG_PATH_PATTERN = re.compile(r"/(live|live-news|live-updates|liveblog)(/|$)|live-updates", re.IGNORECASE)

ARTICLE_LD_TYPES = {
    "Article", "NewsArticle", "ReportageNewsArticle", "AnalysisNewsArticle", "OpinionNewsArticle",
    "BackgroundNewsArticle", "ReviewNewsArticle", "BlogPosting", "Report",
    "Review",   # WIRED 제품 리뷰 (og:type은 website)
}

PAYWALL_PHRASES = [
    "subscribe to continue", "subscribe to read", "subscription required", "to continue reading",
    "already a subscriber", "sign in to continue", "sign in to read", "log in to continue",
    "create a free account to continue", "this article is for subscribers", "this content is for subscribers",
    "unlock this article", "become a member to read",
]

# 봇 차단 / 챌린지 페이지 흔적 (<title> 또는 HTML 원문)
# 뒤의 세 문구는 rss_news 본문 시험에서 Zacks·MarketWatch·WSJ 차단 페이지로 확인된 문구
BLOCK_TITLE_PHRASES = [
    "access denied", "just a moment", "attention required", "are you a robot", "pardon our interruption",
    "request unsuccessful", "verify you are human", "403 forbidden",
]
BLOCK_HTML_MARKERS = [
    "px-captcha", "captcha-delivery.com", "cf-chl-", "_incapsula_resource",
    "made us think you were a bot", "please enable js and disable any ad blocker",
]

PUBLISHED_DATE_FIELDS = [
    "ld:datePublished", "property:article:published_time", "itemprop:datePublished",
    "name:parsely-pub-date", "name:pubdate", "name:publish-date", "name:sailthru.date",
]

# 언론사별 제외 구역: "호스트/경로" 문자열(www. 제거)에 정규식 검색 (1_google_rss_source_test.ipynb의 SECTION_EXCLUDE_RULES)
# URL만으로 판단하므로 다운로드 전에 거름
SECTION_EXCLUDE_RULES = {
    "nypost.com": [
        r"/betting/",     # 도박 프로모션 (테스트 기간 NY Post 505건 중 25건)
    ],
}

# 아래 판정은 URL·메타데이터로 걸러낼 수 있는 기사 아닌 페이지 → 성공률 분모에서 제외
FILTERABLE_STATUSES = {"EXCLUDED_SECTION", "VIDEO_PAGE", "LIVE_BLOG", "NOT_ARTICLE"}

# ============================================================
# RSS 본문 처리 패턴
# ============================================================
# RSS 본문 끝에 붙는 출처 문구 (기사 본문이 아니므로 제거)
#   WordPress 기본 "The post ... appeared first on ...", Fortune "This story was originally featured on ..." (rss_news 감사에서 확인)
RSS_FOOTER_PATTERN = re.compile(
    r"^(the post .+ appeared first on .+|this (story|article) (was )?originally (appeared|featured|published) .+)$",
    re.IGNORECASE,
)

# RSS 본문이 티저(앞부분만)로 끝나는 흔적. rss_news 감사에서 The Motley Fool이 "Continue reading"으로 끝나는 사례 확인
RSS_TEASER_PATTERN = re.compile(r"(continue reading|read more|keep reading|\[…\]|\[\.\.\.\]|…|\.\.\.)\W*$", re.IGNORECASE)

# 추출한 본문에 섞이면 안 되는 문구 (줄 단위 검사, 검증용)
#   rss_news 매체 검토에서 정제 필요로 기록된 유형: 뉴스레터·앱 홍보, 추천/관련 기사, 출처 푸터
BODY_JUNK_PATTERNS = {
    "newsletter": re.compile(r"sign up for .{0,40}newsletter|subscribe to (our|the) .{0,30}newsletter", re.IGNORECASE),
    "app_promo": re.compile(r"download the .{0,30}app", re.IGNORECASE),
    "related": re.compile(r"^(read more|related|recommended|more from|see also|most popular|trending)\b.{0,60}$", re.IGNORECASE),
    "advertisement": re.compile(r"^advertisement$", re.IGNORECASE),
    "rss_footer": re.compile(r"appeared first on|originally (appeared|featured) on", re.IGNORECASE),
}

In [5]:
# 현재 시점 조사용으로 바꾼 값·추가한 값
MIN_BODY_ARTICLES = 5       # 언론사별 표본이 8건이라 판정 최소 기사 수를 낮춤
MIN_REF_PRECISION = 0.9     # trafilatura 본문 중 JSON-LD 본문에 있는 비율이 이보다 낮으면 JSON-LD 본문 사용

# RSS 본문만 있는 경우 (원문 차단): RSS 본문이 80단어 이상인 기사 비율이 이 이상이고 티저가 드물면 'RSS 본문(미검증)'
RSS_ONLY_MIN_RATE = 0.8
RSS_ONLY_MAX_TEASER = 0.2

# 언론사가 붙인 AI 생성 요약 (LA Times "Perspectives")
BODY_JUNK_PATTERNS["ai_generated"] = re.compile(r"ai-generated content|generated by ai", re.IGNORECASE)

## 0-1. 공통 유틸 (2_rss_history_test.ipynb와 같음)

In [6]:
FEED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 캐시 폴더는 git에 올리지 않음
(CACHE_DIR / ".gitignore").write_text("*\n")


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def now_utc():
    return datetime.now(timezone.utc)


def url_hash(url):
    return hashlib.sha1(url.encode("utf-8")).hexdigest()


def et_day_bounds(start, end):
    # 동부시간 날짜 구간 [start, end] → UTC [시작, 끝)
    start_dt = datetime.combine(start, datetime.min.time(), NEWS_TIMEZONE)
    end_dt = datetime.combine(end + timedelta(days=1), datetime.min.time(), NEWS_TIMEZONE)

    return start_dt.astimezone(timezone.utc), end_dt.astimezone(timezone.utc)


def date_range(start, end):
    days = []
    current = start

    while current <= end:
        days.append(current)
        current += timedelta(days=1)

    return days


def to_feed_dict(feed_ids, extra=None):
    # feed_id 리스트 → {feed_id: 후보 정보 (+추가 정보)} (CANDIDATE_FEEDS 순서 유지)
    feed_ids = set(feed_ids)
    extra = extra or {}

    return {
        feed_id: {**info, **extra.get(feed_id, {})}
        for feed_id, info in CANDIDATE_FEEDS.items()
        if feed_id in feed_ids
    }


def make_session():
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)

    return session


SESSION = make_session()
_last_request_at = {}


def polite_get(url, delay=REQUEST_DELAY, headers=None, retries=MAX_RETRIES):
    # 같은 호스트에 delay초 간격을 두고 요청, 429/5xx/네트워크 오류는 retries번까지 재시도
    # headers: 기본 헤더(RSS용)에 덮어쓸 헤더 (기사 HTML 요청은 HTML_HEADERS)
    host = urlparse(url).hostname

    for attempt in range(retries + 1):
        wait = _last_request_at.get(host, 0) + delay - time.time()

        if wait > 0:
            time.sleep(wait)

        try:
            # archive.org는 Accept에 RSS 형식을 지정하면 요청 제한과 상관없이 바로 429를 돌려줌
            if host == "web.archive.org":
                headers = {"Accept": "*/*"}

            response = SESSION.get(url, timeout=REQUEST_TIMEOUT, headers=headers)
            error = None

        except requests.RequestException as e:
            response = None
            error = f"{type(e).__name__}: {e}"

        finally:
            _last_request_at[host] = time.time()

        retryable = response is None or response.status_code in {429, 500, 502, 503, 504}

        if not retryable or attempt == retries:
            return response, error

        if host == "web.archive.org":
            time.sleep(WAYBACK_BLOCK_WAIT * (attempt + 1))
        else:
            time.sleep(5 * (attempt + 1))


def entry_published(entry):
    # feedparser가 UTC로 정규화한 시각 (published → updated 순)
    for key in ("published_parsed", "updated_parsed"):
        value = entry.get(key)

        if value:
            published = datetime.fromtimestamp(calendar.timegm(value), timezone.utc)
            return published if published >= MIN_VALID_DATE else None

    return None


def parse_feed(content):
    parsed = feedparser.parse(content)
    entries = []

    for entry in parsed.entries:
        published_utc = entry_published(entry)

        entries.append({
            # Atom 피드(The Hill 등)는 제목에 &#8216; 같은 엔티티가 남아 있어 한 번 더 해제
            "title": html.unescape(entry.get("title") or "").strip(),
            "link": entry.get("link"),
            "guid": entry.get("id") or entry.get("link"),
            "published_utc": published_utc,
            "published_et": published_utc.astimezone(NEWS_TIMEZONE) if published_utc else None,
            "summary": html.unescape(entry.get("summary") or "").strip(),
            # RSS 본문 HTML (content:encoded / Atom content). 제공하지 않는 피드는 빈 문자열
            "content": "\n".join(c.get("value") or "" for c in entry.get("content", [])),
            "author": entry.get("author"),
            "tags": [t.get("term") for t in entry.get("tags", []) if t.get("term")],
        })

    return entries


def fetch_feed(url, use_cache=False, delay=REQUEST_DELAY):
    '''
    피드 요청 → {"url", "http_status", "error", "entries", "from_cache"}
    use_cache=True면 200/404 응답을 저장해두고 재요청하지 않음 (과거 날짜 아카이브 전용)
    '''
    path_ok = FEED_CACHE_DIR / f"{url_hash(url)}.xml.gz"
    path_404 = FEED_CACHE_DIR / f"{url_hash(url)}.404"

    if use_cache and path_ok.exists():
        content = gzip.decompress(path_ok.read_bytes())
        return {"url": url, "http_status": 200, "error": None, "entries": parse_feed(content), "from_cache": True}

    if use_cache and path_404.exists():
        return {"url": url, "http_status": 404, "error": None, "entries": [], "from_cache": True}

    response, error = polite_get(url, delay=delay)

    if response is None:
        return {"url": url, "http_status": None, "error": error, "entries": [], "from_cache": False}

    entries = parse_feed(response.content) if response.status_code == 200 else []

    if use_cache and response.status_code == 200:
        path_ok.write_bytes(gzip.compress(response.content))

    if use_cache and response.status_code == 404:
        path_404.touch()

    return {"url": url, "http_status": response.status_code, "error": None, "entries": entries, "from_cache": False}


def add_param(url, key, value):
    return f"{url}{'&' if '?' in url else '?'}{key}={value}"


def entry_times(entries):
    return [e["published_utc"] for e in entries if e["published_utc"]]

In [7]:
HTML_DIR.mkdir(parents=True, exist_ok=True)

HEADING_TAGS = {"h2", "h3", "h4", "h5"}
BLOCK_TAGS = {"p", "li", "blockquote", "pre"} | HEADING_TAGS
DROP_TAGS = ["script", "style", "noscript", "figure", "figcaption", "iframe", "video", "audio"]


def has_junk_class(node):
    classes = (node.get("class") or "").lower().split()

    return any(c.startswith(JUNK_CLASS_PREFIXES) for c in classes)


def html_to_text(fragment, drop_junk=False):
    # 본문 HTML 조각 → 문단 텍스트 (문단마다 줄바꿈). 사진·캡션·스크립트는 제거
    # drop_junk=True면 광고·뉴스레터·관련 기사 등 class가 붙은 요소도 제거 (사이트 본문 영역용)
    if not isinstance(fragment, str) or not fragment.strip():
        return ""

    try:
        tree = lxml_html.fragment_fromstring(fragment, create_parent="div")
    except Exception:
        return ""

    for node in tree.xpath(" | ".join(f".//{tag}" for tag in DROP_TAGS)):
        node.drop_tree()

    if drop_junk:
        for node in [n for n in tree.iter() if isinstance(n.tag, str) and has_junk_class(n)]:
            node.drop_tree()

    # 문단 안에 든 문단(li 안의 p 등)은 바깥 것만 사용. (태그, 텍스트)
    blocks = []

    for node in tree.iter(*BLOCK_TAGS):
        if any(parent.tag in BLOCK_TAGS for parent in node.iterancestors()):
            continue

        text = " ".join(node.text_content().split())

        if drop_junk and text:
            link_chars = sum(len(" ".join(a.text_content().split())) for a in node.iter("a"))

            if link_chars / len(text) >= MAX_LINK_DENSITY:
                continue

        if text:
            blocks.append((node.tag, text))

    # 문단 태그 없이 <br>로만 나뉜 본문
    if not blocks:
        return " ".join(tree.text_content().split())

    # 본문 영역 끝에 문단 없이 남은 소제목 제거 (9to5Mac "Accessory deals on Amazon" 아래 상품 링크가 빠지고 제목만 남는 경우)
    if drop_junk:
        while blocks and blocks[-1][0] in HEADING_TAGS:
            blocks.pop()

    return "\n".join(text for _, text in blocks)


def clean_rss_body(text):
    # RSS 본문 끝 출처 문구 제거
    lines = [line for line in text.split("\n") if not RSS_FOOTER_PATTERN.search(line.strip())]

    return "\n".join(lines).strip()


def rss_body_of(content):
    return clean_rss_body(html_to_text(content))


def is_teaser(text):
    return bool(text) and bool(RSS_TEASER_PATTERN.search(text[-300:]))


def word_count(text):
    return len((text or "").split())


def words_of(text):
    return re.findall(r"[a-z0-9]+", unicodedata.normalize("NFKC", text or "").lower())


def shingles(text, n=SHINGLE_SIZE):
    words = words_of(text)

    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


def share_in(part, whole, n=SHINGLE_SIZE):
    # part의 연속 n단어 묶음 중 whole에도 있는 비율 (part가 n단어보다 짧으면 None)
    part_set = shingles(part, n)

    if not part_set:
        return None

    return round(len(part_set & shingles(whole, n)) / len(part_set), 3)


def junk_hits(text):
    lines = [line.strip() for line in (text or "").split("\n")]

    return sorted(
        name
        for name, pattern in BODY_JUNK_PATTERNS.items()
        if any(pattern.search(line) for line in lines)
    )


def host_of(url):
    host = (urlparse(url or "").hostname or "").lower()

    return host[4:] if host.startswith("www.") else host


def is_excluded_section(url):
    host = host_of(url)
    target = host + urlparse(url).path
    rules = [
        rule
        for site, site_rules in SECTION_EXCLUDE_RULES.items()
        if host == site or host.endswith("." + site)
        for rule in site_rules
    ]

    return any(re.search(rule, target) for rule in rules)


def same_site(url_a, url_b):
    a, b = host_of(url_a), host_of(url_b)

    return a == b or a.endswith("." + b) or b.endswith("." + a)

In [8]:
def html_cache_path(url):
    return HTML_DIR / f"{url_hash(url)}.html.gz"


def load_fetch_log():
    # 같은 URL은 마지막 기록이 우선
    return {record["url"]: record for record in load_jsonl(HTML_FETCH_LOG_PATH)}


def is_retryable_fetch(record):
    # 일시적 오류(연결 오류, 429, 5xx)만 재시도 대상
    return record["fetch_status"] == "REQUEST_ERROR" or record["http_status"] in {429, 500, 502, 503, 504}


def fetch_html(url):
    response, error = polite_get(url, delay=BODY_FETCH_DELAY, headers=HTML_HEADERS, retries=BODY_FETCH_RETRIES)

    record = {
        "url": url,
        "final_url": None,
        "http_status": None,
        "content_type": None,
        "fetch_status": "REQUEST_ERROR",
        "error": error,
        "html_saved": False,
        "fetched_at": now_utc().isoformat(),
    }

    if response is not None:
        content_type = response.headers.get("Content-Type", "").lower()

        record.update(
            final_url=response.url,
            http_status=response.status_code,
            content_type=content_type,
            fetch_status=(
                "HTTP_ERROR" if response.status_code != 200
                else "NOT_HTML" if "html" not in content_type
                else "OK"
            ),
        )

        # 오류 응답이라도 HTML이면 저장 (차단 페이지 확인용)
        if "html" in content_type:
            html_cache_path(url).write_bytes(gzip.compress(response.content))
            record["html_saved"] = True

    append_jsonl(HTML_FETCH_LOG_PATH, record)

    return record


def interleave_by_host(urls):
    # 같은 사이트에 연속 요청하지 않도록 호스트를 번갈아 정렬 (호스트별 대기 시간이 겹치게)
    queues = {}

    for url in dict.fromkeys(urls):
        queues.setdefault(host_of(url), []).append(url)

    ordered = []
    queues = list(queues.values())

    while any(queues):
        for queue in queues:
            if queue:
                ordered.append(queue.pop(0))

    return ordered


def is_host_blocking(record):
    return record["fetch_status"] == "REQUEST_ERROR" or record["http_status"] in {401, 403, 429}


def fetch_html_cached(urls, retry_errors=False, verbose=True):
    log = load_fetch_log()
    urls = interleave_by_host(urls)

    todo = [
        url for url in urls
        if url not in log or (retry_errors and is_retryable_fetch(log[url]))
    ]

    if verbose:
        print(f"URL {len(urls)}개 중 캐시 사용 {len(urls) - len(todo)}개, 새로 다운로드 {len(todo)}개", flush=True)

    # 사이트별 연속 차단 횟수. 한도를 넘으면 그 사이트는 이번 실행에서 더 요청하지 않음 (캐시에 남기지 않음)
    fail_streak = {}
    skipped = {}

    for i, url in enumerate(todo, start=1):
        host = host_of(url)

        if fail_streak.get(host, 0) >= BODY_HOST_FAIL_LIMIT:
            log[url] = {
                "url": url, "final_url": None, "http_status": None, "content_type": None,
                "fetch_status": "SKIPPED_HOST", "error": f"연속 {BODY_HOST_FAIL_LIMIT}회 차단으로 건너뜀",
                "html_saved": False, "fetched_at": None,
            }
            skipped[host] = skipped.get(host, 0) + 1
            continue

        log[url] = fetch_html(url)
        fail_streak[host] = fail_streak.get(host, 0) + 1 if is_host_blocking(log[url]) else 0

        if verbose and (i % 50 == 0 or i == len(todo)):
            print(f"  [{i}/{len(todo)}] {host} {log[url]['fetch_status']} ({log[url]['http_status']})", flush=True)

    if verbose and skipped:
        print("연속 차단으로 건너뛴 사이트:", skipped)

    return {url: log[url] for url in urls}

In [9]:
def read_cached_html(url):
    path = html_cache_path(url)

    if not path.exists():
        return None

    return decode_file(gzip.decompress(path.read_bytes()))


def parse_json_ld(tree):
    # 페이지의 JSON-LD 최상위 객체(+@graph)를 평탄화해서 리스트로 반환
    items = []

    for node in tree.xpath('//script[@type="application/ld+json"]'):
        try:
            data = json.loads(node.text_content().strip())
        except Exception:
            continue

        stack = [data]

        while stack:
            current = stack.pop()

            if isinstance(current, list):
                stack.extend(current)

            elif isinstance(current, dict):
                items.append(current)

                if "@graph" in current:
                    stack.append(current["@graph"])

    return items


def ld_types_of(item):
    types = item.get("@type", [])

    if isinstance(types, str):
        types = [types]

    return {t for t in types if isinstance(t, str)}


def meta_content(tree, attr, value, lower=True):
    found = tree.xpath(f'//meta[@{attr}="{value}"]/@content')

    if not found:
        return None

    return found[0].strip().lower() if lower else found[0].strip()


def parse_bool(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"

    return None


def to_news_tz(value):
    # 날짜 문자열 → 동부시간 datetime. 시간대 표기가 없으면 동부시간으로 간주
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        dt = date_parser.parse(value.strip())
    except (ValueError, OverflowError):
        return None

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=NEWS_TIMEZONE)

    return dt.astimezone(NEWS_TIMEZONE)


def find_page_date(tree, article_items, fields):
    # fields 순서대로 찾아서 처음 해석되는 날짜
    for field in fields:
        kind, key = field.split(":", 1)

        if kind == "ld":
            values = [item.get(key) for item in article_items]
        else:
            values = tree.xpath(f'//meta[@{kind}="{key}"]/@content')

        for value in values:
            dt = to_news_tz(value)

            if dt:
                return dt

    return None


def site_container_body(tree, final_url):
    # SITE_BODY_CONTAINERS에 등록된 본문 영역의 문단 텍스트 (여러 개면 문서 순서대로 이어붙임)
    container = SITE_BODY_CONTAINERS.get(host_of(final_url))

    if not container:
        return ""

    tag, class_name = container
    nodes = tree.xpath(f"//{tag}[contains(concat(' ', normalize-space(@class), ' '), ' {class_name} ')]")
    node_ids = {id(node) for node in nodes}

    # 영역 안에 같은 영역이 또 있으면 바깥 것만
    nodes = [node for node in nodes if not any(id(parent) in node_ids for parent in node.iterancestors())]

    texts = [html_to_text(lxml_html.tostring(node, encoding="unicode"), drop_junk=True) for node in nodes]

    return "\n".join(t for t in texts if t)


# WordPress 글 데이터(JSON)의 본문 HTML: "content":{"rendered":"..."}
EMBEDDED_CONTENT_PATTERN = re.compile(r'"content":\{"rendered":("(?:[^"\\]|\\.)*")')


def embedded_json_body(raw_html):
    texts = []

    for match in EMBEDDED_CONTENT_PATTERN.finditer(raw_html):
        try:
            texts.append(html_to_text(json.loads(match.group(1))))
        except ValueError:
            continue

    return max(texts, key=len, default="")


def extract_page_features(raw_html, final_url):
    # lxml은 encoding 선언이 있는 str을 받지 못하므로 XML 선언 제거
    tree = lxml_html.fromstring(re.sub(r"^\s*<\?xml[^>]*\?>", "", raw_html))

    page_title = (
        meta_content(tree, "property", "og:title", lower=False)
        or (tree.findtext(".//title") or "").strip()
    )

    og_type = meta_content(tree, "property", "og:type")

    ld_items = parse_json_ld(tree)
    ld_types = set().union(*[ld_types_of(item) for item in ld_items]) if ld_items else set()
    article_items = [item for item in ld_items if ld_types_of(item) & ARTICLE_LD_TYPES]

    # 언론사가 페이지에 넣어둔 기사 본문 (가장 긴 것)
    ld_body = max(
        (" ".join(html.unescape(str(item.get("articleBody") or "")).split()) for item in article_items),
        key=len,
        default="",
    )

    ld_free_values = [
        parse_bool(item.get("isAccessibleForFree"))
        for item in article_items
        if "isAccessibleForFree" in item
    ]
    ld_is_free = None if not ld_free_values else all(v is not False for v in ld_free_values)

    published_dt = find_page_date(tree, article_items, PUBLISHED_DATE_FIELDS)

    path = urlparse(final_url).path
    url_is_video = bool(VIDEO_PATH_PATTERN.search(path) or VIDEO_HOST_PATTERN.search(host_of(final_url)))
    url_is_live = bool(LIVE_BLOG_PATH_PATTERN.search(path))

    title_lower = page_title.lower()
    html_lower = raw_html[:50000].lower()
    is_blocked = (
        any(phrase in title_lower for phrase in BLOCK_TITLE_PHRASES)
        or any(marker in html_lower for marker in BLOCK_HTML_MARKERS)
    )

    trafilatura_body = trafilatura_extract(
        raw_html,
        url=final_url,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    trafilatura_body = (trafilatura_body or "").strip()

    ld_words = word_count(ld_body)
    container_body = site_container_body(tree, final_url)

    # 교차 확인용 기준 본문: 언론사가 넣은 JSON-LD articleBody > 사이트별 본문 영역
    if ld_words >= MIN_BODY_WORDS:
        reference, reference_type = ld_body, "json_ld"
    elif word_count(container_body) >= MIN_BODY_WORDS:
        reference, reference_type = container_body, "container"
    else:
        reference, reference_type = "", None

    # 기준 본문 중 trafilatura 본문에 들어간 비율(누락 확인), trafilatura 본문 중 기준 본문에 있는 비율(오염 확인)
    ref_recall = share_in(reference, trafilatura_body) if reference and trafilatura_body else None
    ref_precision = share_in(trafilatura_body, reference) if reference and trafilatura_body else None

    body, body_method = trafilatura_body, "trafilatura"

    # 사이트별 본문 영역이 있으면 그것을 본문으로 사용 (trafilatura는 교차 확인용)
    #   trafilatura는 NY Post 구독 홍보·뉴스레터 박스·연합 기사 머리글, Ars Technica 사진 설명,
    #   9to5Mac 상품 목록까지 본문에 넣는 경우가 있었음 (2026-11-05 표본 380건 확인)
    if reference_type == "container":
        body, body_method = reference, reference_type

    # trafilatura가 JSON-LD 본문의 일부만 잡았으면 JSON-LD 본문 사용
    elif reference and (ref_recall or 0) < MIN_REF_COVERAGE and word_count(reference) > word_count(trafilatura_body):
        body, body_method = reference, reference_type

    # [2_rss_history_test.ipynb에 없는 규칙] trafilatura 본문에 JSON-LD 본문에 없는 내용이 10% 넘게 섞이면 JSON-LD 본문 사용
    #   LA Times: 제목·오디오 안내·관련 기사 소개·Perplexity AI 요약(Perspectives)이 trafilatura 본문에 섞임 (2026-11-05)
    elif reference_type == "json_ld" and (ref_precision or 0) < MIN_REF_PRECISION:
        body, body_method = reference, reference_type

    # 본문이 HTML 태그가 아니라 페이지 안 JSON에만 있는 경우 (National Review)
    elif word_count(body) < MIN_BODY_WORDS:
        embedded = embedded_json_body(raw_html)

        if word_count(embedded) >= MIN_BODY_WORDS:
            body, body_method = embedded, "embedded_json"

    body_words = word_count(body)
    body_lower = body.lower()

    return {
        "page_title": page_title,
        "og_type": og_type,
        "ld_types": sorted(ld_types),
        "ld_has_article": bool(ld_types & ARTICLE_LD_TYPES),
        "ld_has_video": "VideoObject" in ld_types,
        "ld_has_liveblog": "LiveBlogPosting" in ld_types,
        "ld_is_free": ld_is_free,
        "ld_body": ld_body,
        "ld_words": ld_words,
        "page_published_et": published_dt,
        "url_is_video": url_is_video,
        "url_is_live": url_is_live,
        "is_blocked": is_blocked,
        "paywall_phrase": next((p for p in PAYWALL_PHRASES if p in body_lower), None),
        "body": body,
        "body_method": body_method,
        "trafilatura_words": word_count(trafilatura_body),
        "container_words": word_count(container_body),
        "reference_type": reference_type,
        "ref_recall": ref_recall,
        "ref_precision": ref_precision,
        "body_words": body_words,
        "ld_coverage": round(body_words / ld_words, 3) if ld_words else None,
    }


def classify_page(link, fetch, features):
    # 제외 구역은 URL만으로 판단 (실제 수집에서는 다운로드 전에 거름)
    if is_excluded_section(link):
        return "EXCLUDED_SECTION"

    if fetch is None:
        return "NOT_FETCHED"

    if fetch["fetch_status"] in {"REQUEST_ERROR", "NOT_HTML", "SKIPPED_HOST"}:
        return fetch["fetch_status"]

    if features is not None and features["is_blocked"]:
        return "BLOCKED"

    if fetch["fetch_status"] == "HTTP_ERROR":
        return f"HTTP_{fetch['http_status']}"

    if not same_site(fetch["final_url"], link):
        return "REDIRECTED_AWAY"

    if features is None:
        return "NOT_FETCHED"

    if (
        features["url_is_video"]
        or (features["og_type"] or "").startswith("video")
        or (features["ld_has_video"] and not features["ld_has_article"])
    ):
        return "VIDEO_PAGE"

    if features["url_is_live"] or features["ld_has_liveblog"]:
        return "LIVE_BLOG"

    # 기사 타입 메타데이터(JSON-LD 기사 또는 og:type=article)가 있어야 기사로 인정
    # 1_google_rss_source_test.ipynb는 og:type≠website와 작성 시각 메타도 요구했지만, 여기 링크는 RSS 기사라 작성 시각을 이미 알고 있고
    #   - Mother Jones: JSON-LD·작성 시각 메타가 없는 정상 기사 (og:type=article)
    #   - WIRED 리뷰: og:type=website지만 JSON-LD Review인 정상 기사
    # 가 있어 뺐음 (작성 시각 차이는 date_gap_hours로 따로 확인)
    if not (features["ld_has_article"] or features["og_type"] == "article"):
        return "NOT_ARTICLE"

    if not features["body"]:
        return "EXTRACT_FAILED"

    if features["paywall_phrase"]:
        return "PAYWALL"

    body_is_short = features["body_words"] < MIN_BODY_WORDS
    body_is_partial = features["ld_coverage"] is not None and features["ld_coverage"] < MIN_LD_COVERAGE

    # isAccessibleForFree=false여도 HTML에 본문 전체가 있는 경우가 많아 본문이 짧거나 일부일 때만 유료로 판정
    if features["ld_is_free"] is False and (body_is_short or body_is_partial):
        return "PAYWALL"

    if body_is_short:
        return "TOO_SHORT"

    if body_is_partial:
        return "PARTIAL_BODY"

    return "SUCCESS"


def analyze_page(link, fetch):
    # 저장된 HTML로 (판정, 특징, 분석 오류)
    features = None
    error = None

    if fetch and fetch["html_saved"]:
        raw_html = read_cached_html(link)

        if raw_html:
            try:
                features = extract_page_features(raw_html, fetch["final_url"])
            except Exception as e:
                error = f"{type(e).__name__}: {e}"

    status = "ANALYZE_ERROR" if error else classify_page(link, fetch, features)

    return status, features, error

## 1. 팀원 단위 테스트

네트워크 없이 설정·DB 생성·중복 방지·본문 선택자 등을 확인하는 팀원 테스트입니다.

In [10]:
test_result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py"],
    cwd=TEAM_RSS_DIR, capture_output=True, text=True,
)

print("종료 코드:", test_result.returncode)
print(test_result.stderr.strip().splitlines()[-3:])

종료 코드: 0
['Ran 12 tests in 0.046s', '', 'OK']


## 2. 팀원 수집기 1회 실행

DB가 없을 때만 실행합니다 (RSS 36개 + 매체별 본문 5건). 이미 있으면 그 결과를 사용합니다.

In [11]:
if str((TEAM_RSS_DIR / "src").resolve()) not in sys.path:
    sys.path.insert(0, str((TEAM_RSS_DIR / "src").resolve()))

import rss_pipeline

if not TEAM_DB_PATH.exists():
    TEAM_CHECK_DIR.mkdir(parents=True, exist_ok=True)
    rss_pipeline.collect_once(
        collect=True, db_path=TEAM_DB_PATH, output_dir=TEAM_RUN_DIR,
        body_per_source=TEAM_BODY_PER_SOURCE, delay=TEAM_DELAY,
    )

conn = sqlite3.connect(TEAM_DB_PATH)
team_articles = pd.read_sql("SELECT * FROM articles", conn)
team_attempts = pd.read_sql("SELECT * FROM body_attempts", conn)
team_feeds = pd.read_sql("SELECT * FROM article_feeds", conn)
conn.close()

print(f"기사 {len(team_articles)}건 / 본문 시도 {len(team_attempts)}건 / 기사-피드 연결 {len(team_feeds)}건")

pd.crosstab(team_articles["source_name"], team_articles["body_status"], margins=True)

기사 1260건 / 본문 시도 336건 / 기사-피드 연결 1334건


body_status,body_hold,candidate_review,integration_review,live_review,no_body,no_body_container,pending,poetry_excluded,subscription_review,type_review,video_review,All
source_name,,,,,,,,,,,,
ABC News,0,20,0,0,0,0,42,0,0,15,0,77
Al Jazeera English,0,0,20,0,0,0,8,0,0,0,4,32
Ars Technica,0,0,20,0,0,0,0,0,0,0,0,20
BBC News,0,20,0,0,0,0,56,0,0,19,3,98
Business Insider,0,0,20,0,0,0,2,0,0,0,0,22
CBS News,0,0,20,5,0,0,38,0,0,0,23,86
CNBC,0,18,0,2,1,1,126,0,3,0,0,151
Engadget,0,20,0,0,0,0,1,0,0,0,0,21
Fortune,0,0,17,0,0,0,0,0,0,0,0,17


## 3. 피드 수집·저장 점검

- 발행 시각(`published`)은 RSS 문자열 그대로 저장됩니다. 분석하려면 시간대를 맞춰 변환해야 하므로 변환이 되는지 확인합니다.
- 같은 기사가 여러 피드에 실린 경우 하나로 합쳐졌는지 확인합니다.

In [12]:
def parse_published(value):
    try:
        dt = date_parser.parse(value)
    except (ValueError, OverflowError, TypeError):
        return None

    return dt if dt.tzinfo else None   # 시간대 없는 값은 변환 불가로 봄


team_articles["published_dt"] = team_articles["published"].map(parse_published)

date_check = (
    team_articles.assign(
        empty=team_articles["published"].fillna("").str.strip() == "",
        unparsed=team_articles["published_dt"].isna(),
    )
    .groupby("source_name")
    .agg(articles=("url", "size"), empty_date=("empty", "sum"), unparsed_or_no_tz=("unparsed", "sum"))
)

multi_feed = team_feeds.groupby("url").size()

print(f"여러 피드에 실린 기사 {int((multi_feed > 1).sum())}건 → articles 테이블에는 1행씩 (중복 저장 없음: {team_articles['url'].is_unique})")
date_check[date_check[["empty_date", "unparsed_or_no_tz"]].sum(axis=1) > 0]

여러 피드에 실린 기사 64건 → articles 테이블에는 1행씩 (중복 저장 없음: True)


,articles,empty_date,unparsed_or_no_tz
source_name,,,


### 3-2. 수집 간격이 충분한가

피드는 최신 기사 몇십 건만 보여주므로, 피드 하나가 담는 시간(`cover_hours`)보다 자주 받아야 기사를 놓치지 않습니다.
팀원 수집기 36개 피드를 지금 받아 권장 간격(`poll_minutes` = `cover_hours` × 0.5)을 계산하고, 팀원 문서의 실행 간격(2시간, 10분)으로 충분한지 봅니다.

In [13]:
def check_live_rate(feed_id, info, now):
    result = fetch_feed(info["url"])
    entries = result["entries"]
    times = entry_times(entries)

    row = {
        "feed_id": feed_id,
        "http_status": result["http_status"],
        "error": result["error"],
        "entries": len(entries),
        "newest_et": max(times).astimezone(NEWS_TIMEZONE) if times else None,
    }

    if not entries:
        return {**row, "live_status": "EMPTY" if result["http_status"] == 200 else "HTTP_ERROR"}

    newest_age_hours = (now - max(times)).total_seconds() / 3600 if times else None
    recent = sum(1 for t in times if (now - t).total_seconds() <= RATE_WINDOW_HOURS * 3600)

    # 피드 기사가 모두 최근 24시간 안이면 24시간 기사 수가 피드 크기에 막혀 과소 계산됨
    # → 피드 맨 앞~맨 뒤 기사 사이 시간으로 계산 (Investing.com: 10건이 6분 사이)
    if times and len(times) >= 2 and (now - min(times)).total_seconds() <= RATE_WINDOW_HOURS * 3600:
        span_hours = max((max(times) - min(times)).total_seconds() / 3600, 1 / 60)
        per_hour = (len(times) - 1) / span_hours
    else:
        per_hour = recent / RATE_WINDOW_HOURS

    if newest_age_hours is None or newest_age_hours > STALE_DAYS * 24:
        status = "STALE"
    else:
        status = "OK"

    # 하루 기사가 0~1건이면 발행량이 적어 최대 간격으로 충분
    cover_hours = len(entries) / per_hour if per_hour > 0 else None
    poll_minutes = (
        MAX_POLL_MINUTES if cover_hours is None
        else min(MAX_POLL_MINUTES, max(MIN_POLL_MINUTES, cover_hours * 60 * POLL_SAFETY))
    )

    rss_words = [word_count(rss_body_of(e["content"])) for e in entries]

    return {
        **row,
        "live_status": status,
        "newest_age_hours": round(newest_age_hours, 1) if newest_age_hours is not None else None,
        "per_day": round(per_hour * 24, 1),
        "cover_hours": round(cover_hours, 1) if cover_hours else None,
        "poll_minutes": round(poll_minutes),
        "miss_risk": cover_hours is not None and cover_hours * 60 * POLL_SAFETY < MIN_POLL_MINUTES,
        "rss_body_words": float(pd.Series(rss_words).median()),
        "entries_list": entries,
    }


TEAM_FEED_LIST = rss_pipeline.load_feeds(config_dir=TEAM_RSS_DIR / "config")
rate_now = now_utc()

team_rate_df = pd.DataFrame([
    {"source_name": f["source_name"], "feed_name": f["feed_name"],
     **{k: v for k, v in check_live_rate(f["feed_name"], {"url": f["feed_url"]}, rate_now).items() if k != "entries_list"}}
    for f in TEAM_FEED_LIST
])

for label, minutes in TEAM_INTERVALS.items():
    team_rate_df[f"enough_{label}"] = team_rate_df["poll_minutes"] >= minutes

team_rate_df.to_csv(TEAM_RATE_PATH, index=False, encoding="utf-8-sig")

team_rate_df[[
    "source_name", "feed_name", "live_status", "entries", "per_day", "cover_hours", "poll_minutes",
    "enough_2시간", "enough_10분", "rss_body_words",
]].sort_values("poll_minutes")

,source_name,feed_name,live_status,entries,per_day,cover_hours,poll_minutes,enough_2시간,enough_10분,rss_body_words
13,Business Insider,legacy_atom,OK,20,230.7,2.1,62,False,True,611.5
7,The Motley Fool,headlines,OK,50,278.2,4.3,129,True,True,123.0
6,Fortune,all,OK,10,47.6,5.0,151,True,True,662.0
34,Al Jazeera English,all,OK,25,109.8,5.5,164,True,True,0.0
0,CNBC,top,OK,30,38.3,18.8,360,True,True,0.0
21,TechCrunch,all,OK,20,20.7,23.2,360,True,True,0.0
22,NPR,news,OK,10,14.2,16.9,360,True,True,32.5
23,NPR,business,OK,10,2.0,120.0,360,True,True,32.0
24,ProPublica,main,OK,20,5.0,96.0,360,True,True,1502.5
25,IEEE Spectrum,all,OK,30,0.0,NaN,360,True,True,0.0


### 3-3. 매체 간 중복 기사

ABC·CBS·NBC 등은 AP 통신 기사를 그대로 싣는 경우가 있어, 매체가 달라도 같은 기사일 수 있습니다. 팀원 수집기는 URL로만 중복을 합칩니다.
정규화한 제목이 같은 기사가 여러 매체에 있는지 확인합니다.

In [14]:
team_articles["norm_title"] = team_articles["title"].map(lambda t: " ".join(words_of(html.unescape(t or ""))))

cross_dup = (
    team_articles[team_articles["norm_title"] != ""]
    .groupby("norm_title")
    .filter(lambda g: g["source_name"].nunique() > 1)
    .sort_values("norm_title")
)

wire_urls = team_articles["url"].str.contains("/wireStory/|/ap-|apnews", case=False, regex=True)

print(f"여러 매체에 같은 제목으로 실린 기사: {cross_dup['norm_title'].nunique()}개 제목 / {len(cross_dup)}건")
print(f"URL에 통신사 기사 표시(wireStory 등)가 있는 기사: {int(wire_urls.sum())}건 ({team_articles.loc[wire_urls, 'source_name'].value_counts().to_dict()})")

cross_dup[["source_name", "title", "url"]].head(20)

여러 매체에 같은 제목으로 실린 기사: 0개 제목 / 0건
URL에 통신사 기사 표시(wireStory 등)가 있는 기사: 53건 ({'ABC News': 53})


,source_name,title,url


## 4. 본문 대조

팀원 수집기가 본문을 시도한 기사를 독립적으로 다시 받아(Chrome User-Agent, 캐시) `2_rss_history_test.ipynb`와 같은 방식으로 추출하고 대조합니다.

In [15]:
attempt_df = pd.DataFrame([
    {**json.loads(row["result_json"]), "attempt_status": row["status"]}
    for row in team_attempts.to_dict("records")
])
attempt_df = attempt_df.merge(
    team_articles[["url", "title", "rss_summary"]].rename(columns={"title": "rss_title"}),
    on="url", how="left",
)

check_fetch_map = fetch_html_cached(attempt_df["url"].tolist(), retry_errors=True)


def compare_team_body(row):
    fetch = check_fetch_map.get(row["url"])
    status, features, error = analyze_page(row["url"], fetch)
    features = features or {}

    team_body = row.get("body_clean") or ""
    ld_body = features.get("ld_body") or ""
    our_body = features.get("body", "") if status == "SUCCESS" else ""

    # 기준 본문: JSON-LD articleBody > 독립 추출 본문
    if word_count(ld_body) >= MIN_BODY_WORDS:
        reference, reference_type = ld_body, "json_ld"
    elif our_body:
        reference, reference_type = our_body, features.get("body_method")
    else:
        reference, reference_type = "", None

    recall = share_in(reference, team_body) if reference and team_body else None
    precision = share_in(team_body, reference) if reference and team_body else None

    team_words = word_count(team_body)
    paragraphs = [p.strip() for p in team_body.split("\n") if p.strip()]
    dup_rate = 1 - len(set(paragraphs)) / len(paragraphs) if paragraphs else 0.0
    length_ratio = team_words / word_count(reference) if reference else None

    if team_words == 0:
        verdict = "NO_BODY"
    elif dup_rate > MAX_DUP_PARAGRAPH_RATE or (length_ratio and length_ratio > MAX_LENGTH_RATIO):
        verdict = "DUPLICATED_OR_BLOATED"
    elif status in {"BLOCKED", "PAYWALL", "VIDEO_PAGE", "LIVE_BLOG", "NOT_ARTICLE"} or status.startswith("HTTP_"):
        verdict = f"SAVED_BUT_{status}"   # 팀원은 본문으로 저장했지만 독립 판정은 기사 본문이 아님
    elif reference_type is None:
        verdict = "NO_REFERENCE"
    elif recall < MIN_TEAM_RECALL:
        verdict = "MISSING_PART"
    elif precision < MIN_TEAM_PRECISION:
        verdict = "EXTRA_TEXT"
    elif team_words < MIN_BODY_WORDS:
        verdict = "SHORT"
    else:
        verdict = "OK"

    return {
        "source_name": row["source_name"],
        "url": row["url"],
        "rss_title": row["rss_title"],
        "team_status": row["body_status"],
        "team_method": row.get("extraction_method"),
        "team_http": row.get("http_status"),
        "our_http": fetch["http_status"] if fetch else None,
        "our_status": status,
        "team_words": team_words,
        "our_words": word_count(our_body),
        "ld_words": word_count(ld_body),
        "reference_type": reference_type,
        "team_recall": recall,
        "team_precision": precision,
        "team_junk": junk_hits(team_body),
        "dup_paragraph_rate": round(dup_rate, 3),
        "length_ratio": round(length_ratio, 2) if length_ratio else None,
        "team_kind": row.get("body_kind"),
        "verdict": verdict,
        "team_body": team_body,
        "reference_body": reference,
    }


team_check_df = pd.DataFrame([compare_team_body(row) for row in attempt_df.to_dict("records")])
team_check_df.to_parquet(TEAM_CHECK_PATH, index=False)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

pd.crosstab(team_check_df["source_name"], team_check_df["verdict"], margins=True)

URL 336개 중 캐시 사용 336개, 새로 다운로드 0개


verdict,DUPLICATED_OR_BLOATED,EXTRA_TEXT,MISSING_PART,NO_BODY,OK,All
source_name,,,,,,
ABC News,0,0,0,0,20,20
Al Jazeera English,0,0,0,0,20,20
Ars Technica,0,0,0,0,20,20
BBC News,0,1,0,0,19,20
Business Insider,0,2,0,0,18,20
CBS News,0,0,0,0,20,20
CNBC,0,0,0,2,18,20
Engadget,0,0,0,0,20,20
Fortune,0,0,0,0,17,17


In [16]:
team_summary = (
    team_check_df
    .assign(
        ok=lambda d: d["verdict"] == "OK",
        has_junk=lambda d: d["team_junk"].str.len() > 0,
    )
    .groupby("source_name")
    .agg(
        attempts=("url", "size"),
        ok=("ok", "sum"),
        team_words_med=("team_words", "median"),
        ref_words_med=("reference_body", lambda x: x.map(word_count).median()),
        recall_med=("team_recall", "median"),
        recall_min=("team_recall", "min"),
        precision_med=("team_precision", "median"),
        precision_min=("team_precision", "min"),
        junk=("has_junk", "sum"),
        reference=("reference_type", lambda x: ", ".join(sorted(set(x.dropna())))),
    )
    .reset_index()
)
team_summary.to_csv(TEAM_SUMMARY_PATH, index=False, encoding="utf-8-sig")

team_summary.round(3).sort_values("ok")

,source_name,attempts,ok,team_words_med,ref_words_med,recall_med,recall_min,precision_med,precision_min,junk,reference
12,NPR,20,14,1370.0,936.5,0.975,0.093,0.910,0.218,2,trafilatura
8,Fortune,17,17,777.0,768.0,1.000,0.985,1.000,0.915,1,trafilatura
11,NBC News,19,17,842.0,800.0,1.000,0.938,0.971,0.744,4,json_ld
10,IEEE Spectrum,20,17,1148.5,1133.0,0.962,0.866,0.971,0.910,3,trafilatura
4,Business Insider,20,18,598.5,558.5,0.990,0.908,0.932,0.479,1,json_ld
6,CNBC,20,18,488.5,494.0,0.996,0.978,0.979,0.860,4,trafilatura
13,ProPublica,20,19,1510.5,1488.5,1.000,0.994,1.000,0.982,6,trafilatura
9,Fox Business,20,19,533.5,535.0,0.998,0.798,1.000,0.934,7,json_ld
16,The Motley Fool,20,19,566.0,525.0,0.988,0.973,0.960,0.913,0,trafilatura
3,BBC News,20,19,930.0,889.5,0.998,0.973,0.946,0.762,4,trafilatura


### 4-2. 다른 방식과 비교

같은 기사에서 네 가지 방식으로 본문을 구해 비교합니다.

| 방식 | 내용 |
|---|---|
| `team` | 팀원 수집기 (매체별 CSS 선택자 + 확인된 문구 삭제) |
| `traf` | trafilatura 자동 추출 (`favor_precision=True`) |
| `ld` | 언론사가 페이지에 넣은 JSON-LD `articleBody` (있는 매체만) |
| `rss` | RSS 피드의 본문(`content`) |

- **JSON-LD 기준 비교**: `ld`가 있는 매체에서 각 방식의 recall(빠진 게 없나)·precision(섞인 게 없나)
- **잡문구 줄 수**: 앞의 검증에서 실제로 확인된 유형(관련 기사, 뉴스레터·앱 홍보, 사진 출처, 제보 안내, 기자 소개, 주가 위젯 등)을 같은 패턴으로 두 방식에 똑같이 적용
- **RSS 전문 여부**: 팀원 본문의 90% 이상이 RSS 본문에 있으면 원문 요청 없이 RSS만으로 충분
- **숨김 텍스트**: 화면에 안 보이는 영역(`hidden`, `aria-hidden`, `display:none`)의 글을 본문으로 가져왔는지 (유료 기사 숨김 본문 위험)

In [17]:
JUNK_LINE_PATTERNS = {
    "related": re.compile(r"^(more on this story|related|recommended|read more|more from|most read|popular reads|see also|trending)\b", re.IGNORECASE),
    "newsletter": re.compile(r"sign up for|newsletter", re.IGNORECASE),
    "promo": re.compile(r"want more .{0,40} in your news feed|add .{0,20} in google|download the .{0,20}app|click here to", re.IGNORECASE),
    "photo_credit": re.compile(r"(getty images|ap photo|afp via getty)\W*\)?\s*$|^image source:", re.IGNORECASE),
    "contact": re.compile(r"^(get in touch|have you been affected|do you live near|help us report|have a tip)", re.IGNORECASE),
    "author_bio": re.compile(r"\bis an? (senior |deputy |national |chief )?(reporter|editor|correspondent|writer|producer|photo editor|photo director|director)\b", re.IGNORECASE),
    "contributor": re.compile(r"contributed to this report", re.IGNORECASE),
    "ticker_widget": re.compile(r"^[A-Z]{1,5} - .{2,60}$|as of \d{1,2}:\d{2} [ap]m et|^key data points$", re.IGNORECASE),
    "section_label": re.compile(r"^(politics|business|elections|book reviews|life kit|world|technology|in:.*)$", re.IGNORECASE),
}


def junk_lines(text):
    found = []

    for line in (text or "").split("\n"):
        line = line.strip()

        for name, pattern in JUNK_LINE_PATTERNS.items():
            if line and pattern.search(line):
                found.append(name)
                break

    return found


def hidden_text_of(tree):
    nodes = tree.xpath('//*[@hidden or @aria-hidden="true" or contains(translate(@style, " ", ""), "display:none")]')
    node_ids = {id(n) for n in nodes}
    nodes = [n for n in nodes if not any(id(a) in node_ids for a in n.iterancestors())]

    return " ".join(" ".join(n.text_content().split()) for n in nodes if n.tag not in ("script", "style"))


team_rss_content = team_articles.set_index("url")["rss_content"].to_dict()


def compare_methods(row):
    fetch = check_fetch_map.get(row["url"])
    raw = read_cached_html(row["url"]) if fetch and fetch["html_saved"] else None

    if not raw or fetch["http_status"] != 200:
        return None

    tree = lxml_html.fromstring(re.sub(r"^\s*<\?xml[^>]*\?>", "", raw))
    features = extract_page_features(raw, fetch["final_url"])

    bodies = {
        "team": row["team_body"],
        "traf": (trafilatura_extract(raw, url=fetch["final_url"], include_comments=False, include_tables=False, favor_precision=True) or "").strip(),
        "ld": features["ld_body"],
        "rss": rss_body_of("\n".join(json.loads(team_rss_content.get(row["url"]) or "[]"))),
    }
    ld = bodies["ld"] if word_count(bodies["ld"]) >= MIN_BODY_WORDS else ""
    hidden = hidden_text_of(tree)

    result = {
        "source_name": row["source_name"],
        "url": row["url"],
        **{f"{m}_words": word_count(b) for m, b in bodies.items()},
        "has_ld": bool(ld),
        "team_vs_traf_recall": share_in(bodies["traf"], bodies["team"]) if bodies["team"] and bodies["traf"] else None,
        "team_vs_traf_precision": share_in(bodies["team"], bodies["traf"]) if bodies["team"] and bodies["traf"] else None,
        "rss_covers_team": share_in(bodies["team"], bodies["rss"]) if bodies["team"] and bodies["rss"] else None,
        "team_junk_lines": len(junk_lines(bodies["team"])),
        "traf_junk_lines": len(junk_lines(bodies["traf"])),
        "traf_from_hidden": bool(bodies["traf"]) and word_count(hidden) >= MIN_BODY_WORDS and (share_in(bodies["traf"], hidden) or 0) >= 0.5,
        "team_from_hidden": bool(bodies["team"]) and word_count(hidden) >= MIN_BODY_WORDS and (share_in(bodies["team"], hidden) or 0) >= 0.5,
    }

    for m in ["team", "traf", "rss"]:
        result[f"{m}_ld_recall"] = share_in(ld, bodies[m]) if ld and bodies[m] else None
        result[f"{m}_ld_precision"] = share_in(bodies[m], ld) if ld and bodies[m] else None

    return result


methods_df = pd.DataFrame([r for r in (compare_methods(row) for row in team_check_df.to_dict("records")) if r])
methods_df.to_parquet(TEAM_METHODS_PATH, index=False)

print(f"비교한 기사 {len(methods_df)}건")

비교한 기사 336건


In [18]:
# (1) JSON-LD가 있는 매체: 각 방식의 JSON-LD 대비 recall / precision (중앙값)
ld_rows = methods_df[methods_df["has_ld"]]

ld_compare = ld_rows.groupby("source_name").agg(
    articles=("url", "size"),
    team_recall=("team_ld_recall", "median"), team_precision=("team_ld_precision", "median"),
    traf_recall=("traf_ld_recall", "median"), traf_precision=("traf_ld_precision", "median"),
    rss_recall=("rss_ld_recall", "median"), rss_precision=("rss_ld_precision", "median"),
)

ld_compare.round(3)

,articles,team_recall,team_precision,traf_recall,traf_precision,rss_recall,rss_precision
source_name,,,,,,,
Business Insider,20,0.990,0.932,1.000,1.000,1.0,0.917
CBS News,20,1.000,0.982,0.952,0.972,NaN,NaN
Fox Business,20,0.998,1.000,0.928,0.977,1.0,1.000
NBC News,19,1.000,0.971,1.000,1.000,0.0,0.000


In [19]:
# (2) 전체 매체: 팀원 vs trafilatura 일치도, 잡문구 줄 수, RSS 전문 여부, 숨김 텍스트
method_summary = methods_df.groupby("source_name").agg(
    articles=("url", "size"),
    team_words=("team_words", "median"),
    traf_words=("traf_words", "median"),
    rss_words=("rss_words", "median"),
    traf_in_team=("team_vs_traf_recall", "median"),
    team_in_traf=("team_vs_traf_precision", "median"),
    team_junk_per_article=("team_junk_lines", "mean"),
    traf_junk_per_article=("traf_junk_lines", "mean"),
    rss_full_rate=("rss_covers_team", lambda x: (x >= 0.9).mean() if x.notna().any() else None),
    traf_hidden=("traf_from_hidden", "sum"),
    team_hidden=("team_from_hidden", "sum"),
).round(3)

method_summary.sort_values("team_junk_per_article", ascending=False)

,articles,team_words,traf_words,rss_words,traf_in_team,team_in_traf,team_junk_per_article,traf_junk_per_article,rss_full_rate,traf_hidden,team_hidden
source_name,,,,,,,,,,,
The Motley Fool,20,566.0,525.0,116.0,0.988,0.960,3.550,0.250,0.000,0,0
Business Insider,20,598.5,562.5,611.5,0.992,0.940,2.050,0.150,0.850,0,0
CBS News,20,552.5,534.0,0.0,0.976,0.937,1.250,0.000,NaN,0,0
NPR,20,1370.0,936.5,32.0,0.975,0.910,1.250,0.350,0.000,0,0
BBC News,20,930.0,889.5,0.0,0.998,0.946,1.200,0.350,NaN,0,0
NBC News,19,842.0,806.0,17.0,1.000,0.962,0.789,0.105,0.000,0,0
ProPublica,20,1510.5,1488.5,1502.5,1.000,1.000,0.650,0.300,0.950,17,17
Fox Business,20,533.5,507.0,534.0,0.978,0.933,0.500,0.350,1.000,0,0
CNBC,20,488.5,494.0,0.0,0.996,0.979,0.350,0.100,NaN,1,0


In [20]:
# 잡문구 유형별 줄 수 (팀원 / trafilatura)
junk_type_df = pd.DataFrame(
    [{"source_name": r["source_name"], "method": "team", "type": t} for r in team_check_df.to_dict("records") for t in junk_lines(r["team_body"])]
)

if len(junk_type_df):
    display(pd.crosstab(junk_type_df["source_name"], junk_type_df["type"], margins=True))

# 숨김 텍스트를 본문으로 가져온 기사
methods_df[methods_df["traf_from_hidden"] | methods_df["team_from_hidden"]][["source_name", "url", "team_words", "traf_words", "traf_from_hidden", "team_from_hidden"]]

type,author_bio,contact,contributor,newsletter,photo_credit,promo,related,section_label,ticker_widget,All
source_name,,,,,,,,,,
ABC News,0,0,2,0,0,0,0,0,0,2
Ars Technica,0,0,0,1,0,0,0,1,0,2
BBC News,0,8,0,5,1,0,10,0,0,24
Business Insider,0,0,0,3,0,38,0,0,0,41
CBS News,0,0,5,0,0,0,0,20,0,25
CNBC,0,0,2,0,0,1,4,0,0,7
Fortune,0,0,0,3,0,0,1,0,0,4
Fox Business,0,0,3,0,0,6,1,0,0,10
IEEE Spectrum,0,0,0,1,1,0,4,0,0,6


,source_name,url,team_words,traf_words,traf_from_hidden,team_from_hidden
3,CNBC,https://www.cnbc.com/2026/11/04/democratic-dat...,0,988,True,False
50,ProPublica,https://www.propublica.org/article/afriforum-g...,1948,1930,True,True
51,ProPublica,https://www.propublica.org/article/safesport-c...,749,746,True,True
52,ProPublica,https://www.propublica.org/article/vermont-cle...,1302,1286,True,True
53,ProPublica,https://www.propublica.org/article/lake-champl...,3110,3096,True,True
232,ProPublica,https://www.propublica.org/article/propublica-...,1321,1311,True,True
233,ProPublica,https://www.propublica.org/article/ruidoso-new...,1038,1036,True,True
234,ProPublica,https://www.propublica.org/article/american-ci...,393,387,True,True
235,ProPublica,https://www.propublica.org/article/five-voting...,2796,2771,True,True
236,ProPublica,https://www.propublica.org/article/border-wall...,1777,1773,True,True


### 4-3. NPR 대본 중복 원인 재현

NPR 대본 페이지(`/transcripts/...`)를 팀원 함수 `find_transcript_text`가 쓰는 `html.parser`와 `lxml`로 각각 읽어 비교합니다.

In [21]:
from npr_ieee_rules import find_transcript_link, find_transcript_text
from bs4 import BeautifulSoup

npr_transcript_rows = team_check_df[(team_check_df["source_name"] == "NPR") & (team_check_df["team_kind"] == "public_transcript")]
print(f"NPR 대본으로 저장된 기사 {len(npr_transcript_rows)}건 / 단어 수: {npr_transcript_rows['team_words'].tolist()}")

sample_url = npr_transcript_rows["url"].iloc[0]
article_soup = BeautifulSoup(read_cached_html(sample_url), "html.parser")
transcript_url = find_transcript_link(article_soup, sample_url)
transcript_html = polite_get(transcript_url, delay=BODY_FETCH_DELAY, headers=HTML_HEADERS)[0].content

for parser in ["html.parser", "lxml"]:
    soup = BeautifulSoup(transcript_html, parser)
    node = max(soup.select(".transcript"), key=lambda n: len(n.select("p")))
    nested = sum(1 for p in node.select("p") if p.find_parent("p"))
    candidate = find_transcript_text(soup)

    print(
        f"{parser:12s} 문단 {len(node.select('p'))}개 중 다른 문단 안에 들어간 문단 {nested}개 "
        f"→ 팀원 함수 결과 {word_count(candidate['text'])}단어"
    )

NPR 대본으로 저장된 기사 6건 / 단어 수: [15208, 39458, 17414, 13332, 872156, 19612]


html.parser  문단 41개 중 다른 문단 안에 들어간 문단 40개 → 팀원 함수 결과 15208단어
lxml         문단 41개 중 다른 문단 안에 들어간 문단 0개 → 팀원 함수 결과 651단어


## 5. 문제 사례 눈으로 확인

대조 결과가 `OK`가 아닌 기사의 팀원 본문과 기준 본문을 나란히 봅니다. 기준 본문 쪽이 틀린 경우도 있으므로 판정 전에 직접 확인합니다.

In [22]:
def contains_line(line, text):
    # 줄이 다른 본문에 들어있는지 (5단어 이상은 겹침 50% 이상, 짧은 줄은 문자열 포함)
    if word_count(line) >= SHINGLE_SIZE:
        return (share_in(line, text) or 0) >= 0.5

    return " ".join(line.split()).lower() in " ".join(text.split()).lower()


def show_diff(row, chars=250):
    team_body, reference = row["team_body"], row["reference_body"]

    print(f"[{row['verdict']}] {row['source_name']} | {row['rss_title']}")
    print(f"URL: {row['url']}")
    print(
        f"팀원 {row['team_words']}단어 ({row['team_status']}) | 기준 {word_count(reference)}단어 ({row['reference_type']}) "
        f"| recall {row['team_recall']} precision {row['team_precision']} | 독립 판정 {row['our_status']} | 오염 {list(row['team_junk'])}"
    )

    # 기준 본문에는 있는데 팀원 본문에 없는 문장 / 팀원 본문에만 있는 문장
    missing = [s for s in re.split(r"(?<=[.!?])\s+|\n", reference) if word_count(s) >= 6 and not contains_line(s, team_body)]
    extra = [s for s in team_body.split("\n") if s.strip() and reference and not contains_line(s, reference)]

    print(f"── 팀원 본문에서 빠진 문장 {len(missing)}개 ──")
    for s in missing[:5]:
        print("  -", s[:chars])

    print(f"── 팀원 본문에만 있는 줄 {len(extra)}개 ──")
    for s in extra[:8]:
        print("  +", s[:chars])

    print("\n" + "=" * 100 + "\n")


for row in team_check_df[team_check_df["verdict"] != "OK"].sort_values(["verdict", "source_name"]).to_dict("records"):
    show_diff(row)

[DUPLICATED_OR_BLOATED] NPR | It was a big night for doctors to win seats in Congress, riding the blue wave
URL: https://www.npr.org/2026/11/04/nx-s1-6017757/democrats-republicans-doctors-acton-andrews-blue-wave
팀원 15208단어 (candidate_review) | 기준 1672단어 (trafilatura) | recall 0.093 precision 0.218 | 독립 판정 SUCCESS | 오염 []


── 팀원 본문에서 빠진 문장 80개 ──
  - It was a big night for doctors to win seats in Congress, riding the blue wave
  - Senate will soon have a practicing physician who's a Democrat for the first time since 1938, with the election of Dr.
  - Annie Andrews, a pediatrician, in South Carolina.
  - Her win — among many other physician candidates Tuesday night — marks a dramatic shift when it comes to doctors with political power across the country.
  - Historically, doctors have been elected to Congress more often as Republicans, not Democrats.
── 팀원 본문에만 있는 줄 40개 ──
  + MARY LOUISE KELLY, HOST: The last time the Senate included a practicing physician who was a Democrat was back in 1938. That is about to change with Democratic doctors winning big on Tuesday and several Republican doctors losing reelection. NPR's Sele
  + MARY LOUISE KELLY, HOST: The last time the Senate included a practicing physician who was a Democrat was back in 1938. That is about to change with Democratic doctors winning big on

── 팀원 본문에서 빠진 문장 3개 ──
  - Legendary photographer on getting that perfect election night picture
  - On election night, photographers will be in position, ready to snap history-making moments.
  - It's something Pulitzer-prize-winner photographer David Hume Kennerly, has done for decades.
── 팀원 본문에만 있는 줄 0개 ──


[DUPLICATED_OR_BLOATED] NPR | Eastern Kentucky welcomes data centers in hopes they make up for coal losses
URL: https://www.npr.org/2026/11/03/nx-s1-6022442/western-kentucky-welcomes-data-centers-in-hopes-they-make-up-for-coal-losses
팀원 17414단어 (candidate_review) | 기준 751단어 (trafilatura) | recall 0.923 precision 0.868 | 독립 판정 SUCCESS | 오염 []


── 팀원 본문에서 빠진 문장 4개 ──
  - Eastern Kentucky welcomes data centers in hopes they make up for coal losses
  - In Kentucky's former coal country, data centers are a welcome sight as people hope they will bring in money.
  - A previous headline incorrectly said data centers are being welcomed in Western Kentucky.
  - In fact, they are being welcomed in Eastern Kentucky.
── 팀원 본문에만 있는 줄 0개 ──


[DUPLICATED_OR_BLOATED] NPR | Chicago entrepreneur turns empty downtown office space into vertical gardens
URL: https://www.npr.org/2026/11/03/nx-s1-5944524-e1/chicago-entrepreneur-turns-empty-downtown-office-space-into-vertical-gardens
팀원 13332단어 (candidate_review) | 기준 672단어 (trafilatura) | recall 0.944 precision 0.874 | 독립 판정 SUCCESS | 오염 []
── 팀원 본문에서 빠진 문장 3개 ──
  - Chicago entrepreneur turns empty downtown office space into vertical gardens
  - The pandemic changed how and where the world works, leaving behind a lot of empty office space.
  - One entrepreneur is starting to plant the seeds of a

── 팀원 본문에서 빠진 문장 3개 ──
  - When caring for aging family members becomes cost prohibitive
  - AARP reports family caregivers are spending 60% more than they were five years ago.
  - Journalist Paula Span discusses the rising costs of caregiving and senior living, and the options available to us.
── 팀원 본문에만 있는 줄 0개 ──


[DUPLICATED_OR_BLOATED] NPR | Mixed economic forecast weighs heavy ahead of midterms
URL: https://www.npr.org/2026/10/31/nx-s1-6022285/mixed-economic-forecast-weighs-heavy-ahead-of-midterms
팀원 19612단어 (candidate_review) | 기준 879단어 (trafilatura) | recall 0.953 precision 0.894 | 독립 판정 SUCCESS | 오염 []


── 팀원 본문에서 빠진 문장 4개 ──
  - Mixed economic forecast weighs heavy ahead of midterms
  - There was a mixed Halloween bag of economic news this week.
  - GDP growth is up, but so are prices.
  - For now at least, interest rates are staying the same.
── 팀원 본문에만 있는 줄 0개 ──


[DUPLICATED_OR_BLOATED] The Motley Fool | The Wait Is Over: 2 Historic Artificial Intelligence (AI) Stock Splits Were Just Announced
URL: https://www.fool.com/investing/2026/11/05/the-wait-is-over-2-historic-artificial-intelligence-ai-stock-splits-were-just-announced/?source=iedfolrf0000001
팀원 590단어 (integration_review) | 기준 547단어 (trafilatura) | recall 0.986 precision 0.949 | 독립 판정 SUCCESS | 오염 []
── 팀원 본문에서 빠진 문장 0개 ──
── 팀원 본문에만 있는 줄 6개 ──
  + META - Meta Platforms
  + NASDAQ · As of 4:00 PM ET 11/04/26
  + Key data points
  + SNDK - Sandisk
  + NASDAQ · As of 4:00 PM ET 11/04/26
  + Key data points


[EXTRA_TEXT] BBC News | Altar Stone mystery continues as Stonehenge theory overturned
URL: https://www.bbc.co.uk/news/

── 팀원 본문에서 빠진 문장 0개 ──
── 팀원 본문에만 있는 줄 49개 ──
  + Selected. Our top picks
  + Best overall Farmacy Green Clean Cleansing Balm $37.00 $39.00 Amazon Learn more Best for long-wear makeup Clinique Take The Day Off Cleansing Balm Makeup Remover $undefined $44.00 Ulta Learn more Best for sensitive skin Bioderma Sensibio H2O Micellar
  + Related
  + selected Beauty brands are reformulating their bestsellers without talc — here’s why, according to experts
  + Clinique Take The Day Off Cleansing Balm Makeup Remover
  + Requires minimal rubbing Works on waterproof formulas Lasts a long time
  + Tub packaging Expensive
  + Farmacy Green Clean Cleansing Balm


[MISSING_PART] Fox Business | US stock futures mixed as Democrats score key midterm election wins
URL: https://www.foxbusiness.com/markets/us-stock-futures-mixed-democrats-score-key-midterm-election-wins
팀원 298단어 (integration_review) | 기준 356단어 (json_ld) | recall 0.798 precision 0.949 | 독립 판정 SUCCESS | 오염 []
── 팀원 본문에서 빠진 문장 4개 ──
  - WHAT T

## 6. 검증 결론 (2026-11-05, 팀원 수집기 2회 실행·본문 336건 기준)

### 잘 되는 것
- RSS 36개 피드 2회 모두 실패 0건. 같은 기사가 여러 피드에 있어도 1건으로 저장(64건 확인). 발행 시각 문자열은 모두 시간대 포함으로 변환 가능
- 본문 336건 중 317건이 독립 추출과 일치. 차단·유료 페이지를 본문으로 저장한 경우 0건
- ABC, Al Jazeera, Ars Technica, CBS, Engadget, Guardian, TechCrunch는 20건 모두 일치하고 잡문구도 거의 없음
- 로그인·차단 우회 없음, 실패해도 다음 기사로 진행, 원본 응답·시도 이력 보존

### 문제 (심각한 순)
| 문제 | 근거 | 원인·해결 |
|---|---|---|
| **NPR 대본 본문이 수십 배로 부풀어 저장됨** | 대본으로 저장된 6건이 1.3만~87만 단어, 문단 절반이 중복 | 대본 페이지의 문단 태그를 `html.parser`가 중첩으로 해석해 문단마다 "그 뒤 전체"가 들어감. `lxml` 파서로 바꾸면 651단어로 정상 (4-3 재현) |
| **본문에 잡문구가 남음** | 기사당 잡문구 줄: Motley Fool 3.6, Business Insider 2.1, CBS·NPR·BBC 약 1.2 (trafilatura는 0~0.35) | 표본에서 확인한 문구만 지정해서 지우는 방식이라 처음 보는 형식은 남음 |
| **기사 아닌 글이 본문으로 수집됨** | NBC Select 쇼핑 추천 글(선물용 레고 등), IEEE Spectrum RSS의 외부 웨비나·백서 광고 링크 | URL 유형 필터에 `/select/shopping/` 등이 없음 |
| **수집 간격 안내가 부족함** | Business Insider 피드 20건이 약 2.1시간 분량 → 2시간 간격(기존 문서 권장)이면 누락 | 10분 간격 예시는 충분. 피드별로 1시간 이하 필요 |
| 상태값 부정확 | CNBC Pro 유료 기사가 "구독 제한"이 아닌 "본문 없음" | `isAccessibleForFree: false` 같은 유료 표시를 확인하지 않음 |
| 선정 기준에 신뢰도·금융 관련성 없음 | Motley Fool·Zacks·Al Jazeera·ProPublica 등 포함, CNBC 경제 섹션 일부·LA Times 등 누락 | `3_rss_live_feed_test.ipynb` 참고 |

### 다른 방식과 비교한 결과
- **trafilatura 자동 추출**: 잡문구는 훨씬 적지만(위 표) Ars Technica는 본문의 46%만 가져오고, CNBC Pro 숨김 유료 본문까지 가져옴 → 단독 사용은 불가
- **JSON-LD `articleBody`** (Business Insider·CBS·Fox·NBC만 제공): 언론사가 넣은 깨끗한 본문이라 검증 기준으로 좋음. 다만 목록·표가 빠지는 경우가 있어(BI 추천 공모 글) 본문 대신보다는 대조용이 적합
- **RSS 본문**: Fox Business 100%, ProPublica 95%, Business Insider 85%, Fortune 82%가 원문과 같은 전문 → 이 매체들은 원문 요청 없이 RSS만으로도 대부분 충분 (원문이 막힐 때 대안)

### 권장 개선 (팀원과 공유)
1. NPR 대본 파싱을 `lxml`로 변경 (또는 다른 문단 안에 든 문단 제외)
2. 저장 전 자동 품질 검사: trafilatura·JSON-LD 본문과 겹침 비율, 길이 비율, 반복 문단 비율, 잡문구 패턴 → 기준을 벗어나면 `review` 상태로 표시. 지금은 사람이 표본을 봐야만 문제를 알 수 있음
3. 잡문구 정제를 문구 지정 방식에서 영역·패턴 방식으로 보강 (링크 위주 문단, 관련 기사·뉴스레터 영역 class, 사진 출처 줄, 주가 위젯)
4. URL 유형 필터 추가: NBC·CNBC `/select/`, 매체 도메인이 아닌 외부 링크(on24, wiley 등)
5. 유료 판정에 `isAccessibleForFree`·`premium` 표시 사용
6. 실행 간격은 1시간 이하 (Business Insider 기준). 10분이면 모든 피드 충분
7. 매체 선정 기준에 금융 관련성·신뢰도 추가
